# DAY 10 - Time Series Analysis

**Time Series = Data collected over time.**

Examples:
- Stock prices (daily)
- Monthly sales
- Daily temperature
- Website traffic (hourly)
- COVID cases over time

Why it matters:
- Forecasting future values
- Detecting trends and patterns
- Seasonal analysis
- Business planning

---

## Topics Covered
1. Creating Datetime Data
2. Datetime Indexing
3. Resampling
4. Rolling Windows (Moving Averages)
5. Trend Analysis
6. Seasonality
7. Lag Features
8. Time-based Filtering
9. Real-world Stock Analysis
10. Sales Forecasting Preview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')
print('Libraries loaded!')

---
## 1. Working with Dates and Times

In [ ]:
# Creating datetime objects
today = pd.Timestamp.today()
print('Today:', today)
print('Year:', today.year)
print('Month:', today.month)
print('Day:', today.day)
print('Day Name:', today.day_name())
print('Month Name:', today.month_name())

# Arithmetic
print('\nDate 30 days from now:', today + pd.Timedelta(days=30))
print('Date 6 months ago:', today - pd.DateOffset(months=6))

In [ ]:
# Date ranges — the backbone of time series
daily   = pd.date_range('2024-01-01', periods=10, freq='D')
weekly  = pd.date_range('2024-01-01', periods=6, freq='W')
monthly = pd.date_range('2024-01-01', periods=12, freq='MS')
hourly  = pd.date_range('2024-01-01 09:00', periods=8, freq='H')

print('Daily   :', daily[:5].tolist())
print('Weekly  :', weekly[:3].tolist())
print('Monthly :', monthly[:4].tolist())
print('Hourly  :', hourly[:3].tolist())

print('\nFrequency Codes:')
freqs = [('D', 'Day'), ('W', 'Week'), ('MS', 'Month Start'),
         ('ME', 'Month End'), ('QS', 'Quarter Start'), ('YS', 'Year Start'),
         ('H', 'Hour'), ('T', 'Minute'), ('S', 'Second')]
for code, name in freqs:
    print(f'  {code:4} = {name}')

In [ ]:
# Parse dates from string
date_strings = ['2024-01-15', '15/03/2024', 'Mar 25, 2024', '2024.06.10']
for ds in date_strings:
    try:
        parsed = pd.to_datetime(ds)
        print(f'{ds:20} → {parsed.strftime("%d %B %Y")}')
    except:
        print(f'{ds:20} → Parse Error')

---
## 2. Building a Time Series DataFrame

In [ ]:
# 2 years of daily sales data
np.random.seed(42)
date_range = pd.date_range('2022-01-01', '2023-12-31', freq='D')
n = len(date_range)

# Simulate realistic sales with trend + seasonality + noise
trend     = np.linspace(1000, 1500, n)           # Upward trend
seasonal  = 300 * np.sin(2 * np.pi * np.arange(n) / 365)  # Annual cycle
weekly_s  = 150 * np.sin(2 * np.pi * np.arange(n) / 7)   # Weekly pattern
noise     = np.random.normal(0, 80, n)            # Random noise

sales = trend + seasonal + weekly_s + noise
sales = np.maximum(sales, 100)  # No negative sales

ts = pd.DataFrame({
    'date': date_range,
    'sales': sales.astype(int)
})
ts.set_index('date', inplace=True)

print('Time Series DataFrame:')
print(ts.head(10))
print(f'\nShape: {ts.shape}')
print(f'Date range: {ts.index.min().date()} to {ts.index.max().date()}')
print(f'\nBasic Stats:')
print(ts.describe())

In [ ]:
# Plot full time series
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts.index, ts['sales'], linewidth=1, alpha=0.7, color='steelblue', label='Daily Sales')
ax.set_title('Daily Sales: 2022-2023', fontsize=16)
ax.set_xlabel('Date')
ax.set_ylabel('Sales (Rs)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Resampling — Change Time Frequency

In [ ]:
# Resample to different frequencies
weekly_sales  = ts.resample('W').sum()   # Weekly total
monthly_sales = ts.resample('ME').sum()   # Monthly total
quarter_sales = ts.resample('QE').sum()   # Quarterly total

print('Weekly Sales (last 5 weeks):')
print(weekly_sales.tail())
print('\nMonthly Sales:')
print(monthly_sales)
print('\nQuarterly Sales:')
print(quarter_sales)

In [ ]:
# Compare daily vs monthly
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(ts.index, ts['sales'], linewidth=0.8, alpha=0.6, color='lightblue', label='Daily')
axes[0].plot(monthly_sales.index, monthly_sales['sales'], linewidth=2.5, color='steelblue', label='Monthly Total')
axes[0].set_title('Daily vs Monthly Sales', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(monthly_sales.index, monthly_sales['sales'],
            width=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Monthly Sales Bar Chart', fontsize=14)
axes[1].set_ylabel('Total Sales')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Rolling Windows — Moving Averages

Smooths out noise to see the underlying trend.

In [ ]:
# Calculate moving averages
ts['MA_7']  = ts['sales'].rolling(window=7).mean()    # 7-day MA
ts['MA_30'] = ts['sales'].rolling(window=30).mean()   # 30-day MA
ts['MA_90'] = ts['sales'].rolling(window=90).mean()   # 90-day MA

plt.figure(figsize=(14, 6))
plt.plot(ts.index, ts['sales'], alpha=0.3, color='lightgray', linewidth=1, label='Daily')
plt.plot(ts.index, ts['MA_7'],  color='blue', linewidth=1.5, label='7-Day MA')
plt.plot(ts.index, ts['MA_30'], color='orange', linewidth=2, label='30-Day MA')
plt.plot(ts.index, ts['MA_90'], color='red', linewidth=2.5, label='90-Day MA')

plt.title('Moving Averages: Smoothing Daily Sales', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('The longer the window, the smoother (less noise, more trend)')

In [ ]:
# Rolling statistics
ts['roll_mean'] = ts['sales'].rolling(30).mean()
ts['roll_std']  = ts['sales'].rolling(30).std()
ts['upper_band'] = ts['roll_mean'] + 2 * ts['roll_std']  # Bollinger Bands
ts['lower_band'] = ts['roll_mean'] - 2 * ts['roll_std']

plt.figure(figsize=(14, 6))
plt.plot(ts.index, ts['sales'], alpha=0.5, color='gray', linewidth=1, label='Daily Sales')
plt.plot(ts.index, ts['roll_mean'], color='blue', linewidth=2, label='30-Day Mean')
plt.fill_between(ts.index, ts['upper_band'], ts['lower_band'],
                 alpha=0.15, color='blue', label='±2σ Band')

plt.title('Sales with Bollinger Bands (Used in Trading)', fontsize=16)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Extracting Time Features

In [ ]:
ts_features = ts[['sales']].copy()
ts_features['year']        = ts.index.year
ts_features['month']       = ts.index.month
ts_features['week']        = ts.index.isocalendar().week.astype(int)
ts_features['day_of_week'] = ts.index.dayofweek  # 0=Mon, 6=Sun
ts_features['day_name']    = ts.index.day_name()
ts_features['is_weekend']  = ts.index.dayofweek >= 5
ts_features['quarter']     = ts.index.quarter

print('Time Features:')
print(ts_features.head(10))

In [ ]:
# Day of week analysis
day_avg = ts_features.groupby('day_name')['sales'].mean()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_avg = day_avg.reindex(day_order)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Day of week
colors = ['tomato' if d in ['Saturday','Sunday'] else 'steelblue' for d in day_order]
axes[0].bar(day_order, day_avg.values, color=colors, edgecolor='black')
axes[0].set_title('Avg Sales by Day of Week', fontsize=14)
axes[0].set_ylabel('Avg Daily Sales')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis='y', alpha=0.3)

# Monthly pattern
month_avg = ts_features.groupby('month')['sales'].mean()
axes[1].bar(range(1,13), month_avg.values, color='steelblue', edgecolor='black')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[1].set_title('Avg Sales by Month', fontsize=14)
axes[1].set_ylabel('Avg Daily Sales')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Time-based Filtering

In [ ]:
# Filter by date
june_sales = ts['2023-06']
print('June 2023 Sales:')
print(june_sales.head())
print(f'Total June Sales: {june_sales["sales"].sum():,}')

# Q1 2023
q1_2023 = ts['2023-01':'2023-03']
print(f'\nQ1 2023 Total: {q1_2023["sales"].sum():,}')

# Year comparison
for year in [2022, 2023]:
    yr_data = ts[str(year)]
    print(f'{year} Total Sales: {yr_data["sales"].sum():>12,}  |  Daily Avg: {yr_data["sales"].mean():,.0f}')

In [ ]:
# Year-over-year comparison
yearly_monthly = ts_features.groupby(['year', 'month'])['sales'].sum().unstack(0)

plt.figure(figsize=(13, 5))
for year in yearly_monthly.columns:
    plt.plot(range(1, 13), yearly_monthly[year],
             marker='o', linewidth=2, label=str(year))

plt.xticks(range(1,13), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.title('Year-over-Year Monthly Sales Comparison', fontsize=16)
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.legend(title='Year')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Lag Features & Growth Rate

In [ ]:
# Lag features (previous period's value) — used in forecasting
monthly_ts = ts.resample('ME').sum()[['sales']]

monthly_ts['lag_1']  = monthly_ts['sales'].shift(1)   # Previous month
monthly_ts['lag_3']  = monthly_ts['sales'].shift(3)   # 3 months ago
monthly_ts['lag_12'] = monthly_ts['sales'].shift(12)  # Same month last year

# Growth rate
monthly_ts['mom_growth'] = monthly_ts['sales'].pct_change() * 100  # Month-over-month
monthly_ts['yoy_growth'] = monthly_ts['sales'].pct_change(12) * 100  # Year-over-year

print('Monthly Sales with Lag & Growth:')
print(monthly_ts.tail(15).round(1))

In [ ]:
# Growth rate chart
fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].bar(monthly_ts.index, monthly_ts['sales'],
            color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Monthly Total Sales', fontsize=14)
axes[0].set_ylabel('Total Sales')
axes[0].grid(axis='y', alpha=0.3)

growth = monthly_ts['mom_growth'].dropna()
colors_growth = ['green' if g > 0 else 'red' for g in growth]
axes[1].bar(growth.index, growth.values, color=colors_growth, edgecolor='white', alpha=0.8)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Month-over-Month Growth (%)', fontsize=14)
axes[1].set_ylabel('Growth %')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Anomaly Detection in Time Series

In [ ]:
# Detect anomalies using Z-score on rolling window
rolling_mean = ts['sales'].rolling(30).mean()
rolling_std  = ts['sales'].rolling(30).std()

z_score = (ts['sales'] - rolling_mean) / rolling_std

anomalies = ts[np.abs(z_score) > 2.5]
print(f'Number of anomalies detected: {len(anomalies)}')

plt.figure(figsize=(14, 6))
plt.plot(ts.index, ts['sales'], color='steelblue', linewidth=1, alpha=0.7, label='Sales')
plt.plot(ts.index, rolling_mean, color='orange', linewidth=2, label='30-Day Avg')
plt.fill_between(ts.index,
                 rolling_mean - 2.5 * rolling_std,
                 rolling_mean + 2.5 * rolling_std,
                 alpha=0.1, color='orange', label='Normal Range')
plt.scatter(anomalies.index, anomalies['sales'],
            color='red', s=50, zorder=5, label=f'Anomalies ({len(anomalies)})')
plt.title('Sales Anomaly Detection', fontsize=16)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Real-World: Stock Analysis

In [ ]:
# Simulate 1 year of stock data
np.random.seed(42)
trading_days = pd.bdate_range('2023-01-01', '2023-12-31')  # Business days only
initial_price = 1500  # Rs 1500 starting price
daily_returns = np.random.normal(0.0005, 0.015, len(trading_days))
prices = initial_price * np.cumprod(1 + daily_returns)

stock = pd.DataFrame({
    'close': prices,
    'volume': np.random.randint(100000, 500000, len(trading_days))
}, index=trading_days)

# Technical indicators
stock['MA20'] = stock['close'].rolling(20).mean()   # 20-day MA
stock['MA50'] = stock['close'].rolling(50).mean()   # 50-day MA
stock['daily_return'] = stock['close'].pct_change() * 100

print('Stock Data:')
print(stock.head(5))
print(f'\nStarting Price: Rs {initial_price}')
print(f'Ending Price  : Rs {stock["close"].iloc[-1]:.2f}')
print(f'Annual Return : {((stock["close"].iloc[-1]/initial_price)-1)*100:.2f}%')
print(f'Max Drawdown  : {((stock["close"]/stock["close"].cummax()) - 1).min()*100:.2f}%')
print(f'Volatility    : {stock["daily_return"].std():.2f}% daily')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Price chart with MAs
axes[0].plot(stock.index, stock['close'], color='black', linewidth=1.5, label='Price')
axes[0].plot(stock.index, stock['MA20'], color='blue', linewidth=1.5, label='20-Day MA')
axes[0].plot(stock.index, stock['MA50'], color='red', linewidth=1.5, label='50-Day MA')
axes[0].set_title('Stock Price with Moving Averages', fontsize=15)
axes[0].set_ylabel('Price (Rs)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Volume chart
axes[1].bar(stock.index, stock['volume'], color='steelblue', alpha=0.7, width=0.8)
axes[1].set_title('Trading Volume', fontsize=15)
axes[1].set_ylabel('Volume (Shares)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary — Time Series Functions

| Task | Pandas Code |
|------|-------------|
| Create date range | `pd.date_range(start, periods, freq)` |
| Parse date string | `pd.to_datetime(string)` |
| Set datetime index | `df.set_index('date')` |
| Filter by date | `df['2023-06']` or `df['2023-01':'2023-06']` |
| Resample | `df.resample('ME').sum()` |
| Rolling average | `df['col'].rolling(30).mean()` |
| Shift (lag) | `df['col'].shift(1)` |
| Growth rate | `df['col'].pct_change()` |
| Extract features | `.dt.year`, `.dt.month`, `.dt.day_name()` |

---
## Frequency Reference

| Code | Frequency |
|------|-----------|
| `D` | Daily |
| `W` | Weekly (Sunday) |
| `MS` | Month Start |
| `ME` | Month End |
| `QS` | Quarter Start |
| `YS` | Year Start |
| `H` | Hourly |
| `B` | Business Days |

---
## Homework

1. Create 3 years of monthly revenue data with an upward trend
2. Calculate month-over-month and year-over-year growth
3. Plot moving averages (7-day, 30-day, 90-day)
4. Find which month consistently has highest sales
5. Detect any anomalies in the data